In [1]:
import jax
import jax.numpy as jnp
import optax

from pickle import load

In [3]:
data_path = '/Users/mariana/Documents/projects/Huawei/tdsurv/data/aids-seqs.pkl'

In [6]:
data = load(open(data_path, 'rb'))

In [8]:
data.keys()

dict_keys(['seqs', 'cs', 'ts', 'cols'])

In [11]:
seqs = data['seqs']
cs = data['cs']
ts = data['ts']

In [18]:
seqs[2]

array([[ 0.        ,  1.        ,  1.        ,  1.        , -0.61058817],
       [ 0.        ,  1.        ,  1.        ,  1.        , -0.57296897],
       [ 0.        ,  1.        ,  1.        ,  1.        ,  0.01102591],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]])

In [19]:
ts[2]

3

In [20]:
jnp.eye(3)

Array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

In [32]:
def pad_to(x1, x2):
    a1, a2 = x1.shape
    b1, b2 = x2.shape
    assert b2 >= a2 and b1 >= a1

    miss_cols = b2 - a2
    miss_rows = b1 - a1

    res = jnp.hstack((x1, jnp.zeros((a1, miss_cols))))
    res = jnp.vstack((res, jnp.zeros((miss_rows, b2))))
    return res

In [111]:
def get_target_and_mask(seq, t, c, landmark=False):
    target = jnp.zeros_like(seq)
    if not c:
        target = jnp.eye(t)[::-1]
        target = pad_to(target, seq)
    if landmark:
        mask = jnp.tril(jnp.ones_like(seq), -(t-1))[::-1]
    else:
        mask = jnp.ones((1, t))
        mask = pad_to(mask, seq)
    return target, mask

In [112]:
def tree_get_target_and_mask(tree, landmark=False):
    seq, t, c = tree
    
    target = jnp.zeros_like(seq)
    if not c:
        target = jnp.eye(t)[::-1]
        target = pad_to(target, seq)
    if landmark:
        mask = jnp.tril(jnp.ones_like(seq), -(t-1))[::-1]
    else:
        mask = jnp.ones((1, t))
        mask = pad_to(mask, seq)
    return target, mask

In [113]:
target, mask = get_target_and_mask(seqs[1], ts[1], cs[1], False)

In [138]:
targets = []
masks = []
for seq, t,c in zip(seqs, ts, cs):
    target, mask = get_target_and_mask(seq, t, c, False)
    targets.append(target)
    masks.append(mask)

In [139]:
mask = jnp.stack(masks)
target = jnp.stack(targets)

In [116]:
type(seqs)

numpy.ndarray

In [117]:
optax.sigmoid_binary_cross_entropy(jnp.array(seqs[1]), target) * mask

Array([[0.6931472, 0.6931472, 0.6931472, 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ]],      dtype=float32)

In [79]:
type(seqs)

numpy.ndarray

In [82]:
seqs.shape

(467, 5, 5)

In [88]:
get_tgt = jax.vmap(tree_get_target_and_mask)

In [95]:
ll = [seq[0], seq[0]]

In [96]:
jnp.stack(ll).shape

(2, 5)

In [97]:
mask

Array([[ True,  True,  True, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False]], dtype=bool)

In [118]:
def train_test_split(X, test_size=0.2, random_seed=None):
    # Check if a random seed is provided and set the random key accordingly
    if random_seed is not None:
        rng = jax.random.PRNGKey(random_seed)
    else:
        rng = jax.random.PRNGKey(0)

    # Shuffle the indices of the data
    num_samples = X.shape[0]
    shuffled_indices = jax.random.permutation(rng, jnp.arange(num_samples))

    # Calculate the number of samples in the test set
    num_test_samples = int(num_samples * test_size)

    # Split the shuffled indices into train and test sets
    test_indices = shuffled_indices[:num_test_samples]
    train_indices = shuffled_indices[num_test_samples:]

    # Use the indices to split the data
    X_train = X[train_indices]
    X_test = X[test_indices]

    return X_train, X_test

In [129]:
def batch_generator(X, y, batch_size, shuffle=True, random_seed=None):
    # Check if a random seed is provided and set the random key accordingly
    if random_seed is not None:
        rng = jax.random.PRNGKey(random_seed)
    else:
        rng = jax.random.PRNGKey(0)

    num_samples = X.shape[0]
    
    # Shuffle the data using the same random key for X and y if shuffle is True
    if shuffle:
        rng, subkey = jax.random.split(rng)
        permutation = jax.random.permutation(subkey, jnp.arange(num_samples))
        X = X[permutation]
        y = y[permutation]

    for i in range(0, num_samples, batch_size):
        batch_X = X[i:i + batch_size]
        batch_y = y[i:i + batch_size]
        yield batch_X, batch_y

In [140]:
datagen = batch_generator(seqs, target, 16)

In [141]:
x, y =next(datagen)

In [142]:
x.shape

(16, 5, 5)

In [143]:
y.shape

(16, 5, 5)

In [136]:
jax.random.permutation(jax.random.PRNGKey(0), jnp.arange(10))

Array([2, 7, 9, 6, 0, 8, 1, 3, 4, 5], dtype=int32)

In [165]:
def sinusoid_position_encoding(
    sequence_length: int,
    hidden_size: int,
    max_timescale: float = 1e4,
    add_negative_side: bool = False,
) -> jnp.ndarray:
  """Creates sinusoidal encodings from the original transformer paper.

  The returned values are, for all i < D/2:
    array[pos, i] = sin(pos / (max_timescale^(2*i / D)))
    array[pos, D/2 + i] = cos(pos / (max_timescale^(2*i / D)))

  Args:
    sequence_length: Sequence length.
    hidden_size: Dimension of the positional encoding vectors, D. Should be
      even.
    max_timescale: Maximum timescale for the frequency.
    add_negative_side: Whether to also include the positional encodings for
      negative positions.

  Returns:
    An array of shape [L, D] if add_negative_side is False, else [2 * L, D].
  """
  if hidden_size % 2 != 0:
    raise ValueError(
        'The feature dimension should be even for sin/cos positional encodings.'
    )
  freqs = jnp.arange(0, hidden_size, 2)
  inv_freq = max_timescale ** (-freqs / hidden_size)
  # pos_seq = jnp.arange(
  #     start=-sequence_length if add_negative_side else 0, stop=sequence_length
  # )
  pos_seq = jnp.arange(
      start=0, stop=sequence_length
  )
  sinusoid_inp = jnp.einsum('i,j->ij', pos_seq, inv_freq)
  return jnp.concatenate([jnp.sin(sinusoid_inp), jnp.cos(sinusoid_inp)], axis=-1)

In [166]:
enc= sinusoid_position_encoding(5, 10)

In [169]:
enc

Array([[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
         1.0000000e+00,  1.0000000e+00],
       [ 8.4147102e-01,  1.5782663e-01,  2.5116222e-02,  3.9810603e-03,
         6.3095719e-04,  5.4030234e-01,  9.8746681e-01,  9.9968451e-01,
         9.9999207e-01,  9.9999982e-01],
       [ 9.0929741e-01,  3.1169716e-01,  5.0216597e-02,  7.9620583e-03,
         1.2619141e-03, -4.1614684e-01,  9.5018148e-01,  9.9873835e-01,
         9.9996829e-01,  9.9999923e-01],
       [ 1.4112000e-01,  4.5775455e-01,  7.5285286e-02,  1.1942930e-02,
         1.8928705e-03, -9.8999250e-01,  8.8907862e-01,  9.9716204e-01,
         9.9992865e-01,  9.9999821e-01],
       [-7.5680250e-01,  5.9233773e-01,  1.0030648e-01,  1.5923612e-02,
         2.5238262e-03, -6.5364361e-01,  8.0568981e-01,  9.9495661e-01,
         9.9987322e-01,  9.9999684e-01]], dtype=float32)

In [183]:
def f(inputs):
    linear = hk.Linear(1)
    bias = hk.Bias()
    out = linear(inputs)
    out = bias(out)

    return out

In [184]:
f = hk.without_apply_rng(hk.transform(f))

In [185]:
seqs[0].reshape(1, 5, 5)

array([[[1.        , 0.        , 1.        , 1.        , 0.79578199],
        [1.        , 0.        , 1.        , 1.        , 0.43098522],
        [1.        , 0.        , 1.        , 1.        , 0.59968076],
        [0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ]]])

In [190]:
_key = jax.random.PRNGKey(0)
_some_input= seqs[0].reshape(1, *seqs[0].shape)
params = f.init(_key, _some_input)

In [191]:
params

{'linear': {'w': Array([[-0.5945483 ],
         [-0.77199805],
         [ 0.7720665 ],
         [ 0.4860555 ],
         [-0.4096454 ]], dtype=float32),
  'b': Array([0.], dtype=float32)},
 'bias': {'b': Array([[0.],
         [0.],
         [0.],
         [0.],
         [0.]], dtype=float32)}}

In [182]:
_some_input.shape

(5, 5)

In [195]:
target.argmax(-1).shape

(467, 5)

In [196]:
target.shape

(467, 5, 5)

In [209]:
tgt = target.argmax(1, keepdims=False)
res = out.argmax(1)

NameError: name 'out' is not defined

In [208]:
mask.any(1, keepdims=False)[2]

Array([ True,  True,  True, False, False], dtype=bool)

In [206]:
target[2]

Array([[0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [210]:
100//64

1

In [211]:
math.ceil(1.4)

2

In [37]:
# Trying to make data loading more efficient
import os
import h5py
from math import ceil
from functools import partial

In [12]:

data_path = '/Users/mariana/Documents/projects/Huawei/cloudproject/data/Delays/new_data_b03-09_online/'
task_id = 11

seqs = []
for root, dirs, files in os.walk(data_path):
    for filename in files:
        if filename.endswith('.mat') and 'task_{}'.format(task_id) in filename:
            file_path = os.path.join(root, filename)
            with h5py.File(file_path, 'r') as f:
                if 'traces_self' in f:
                    size = f['traces_self'].shape[0]
                    # Extract the array under the 'traces_self' key
                    t_self = [f['traces_self']
                              [i].reshape(-1, 39) for i in range(size)]
                    seqs.extend(t_self)

In [13]:
len(seqs)

272

In [41]:
@jax.jit
def split_and_pad_last(arr, H=1000):
    t, dim = arr.shape
    n_splits = jnp.ceil(t/H)
    indices = jnp.arange(1, n_splits) * H
    arrs = jnp.array_split(arr, indices_or_sections=indices)
    last = arrs[-1]
    h, _ = last.shape
    if h < H:
        zs = jnp.zeros((H-h, dim))
        last = jnp.concatenate((last, zs))
    arr = jnp.stack(arrs[:-1] + [last])

    ts = t - indices
    ts = jnp.hstack((jnp.array([t]), ts))
    cs = jnp.hstack((jnp.ones_like(indices), jnp.array([0]))).astype(jnp.bool_)
    return arr, ts, cs

In [28]:
sizes = jnp.array([item.shape[0] for item in seqs])

In [22]:
sizes.argmax()

Array(229, dtype=int32)

In [23]:
seq = seqs[229]

In [24]:
seq.shape

(202, 39)

In [29]:
%%time 
arr, t, c = split_and_pad_last(seq, H=100)

CPU times: user 96.4 ms, sys: 7.84 ms, total: 104 ms
Wall time: 102 ms


In [36]:
%%time
horizon = 100
arrs, tss, css = [], [], []
for seq in seqs:
    arr, ts, cs = split_and_pad_last(seq, horizon)
    arrs.append(arr)
    css.append(cs)
    tss.append(ts)
seqs = jnp.vstack(arrs)
ts = jnp.hstack(tss)
cs = jnp.hstack(css)

CPU times: user 932 ms, sys: 26.8 ms, total: 958 ms
Wall time: 956 ms


In [38]:
partial_split = partial(split_and_pad_last, H=horizon)

In [43]:
data_path = '/Users/mariana/Documents/projects/Huawei/SurvanData/mixed_tasks/mixed/H_100/dataset.h5'

In [44]:
data = h5py.File(data_path, 'r')

In [50]:
data['cs']

AttributeError: 'Dataset' object has no attribute 'values'

In [56]:
seqs = jnp.array(data['seqs'])

In [57]:
seqs.shape

(22567, 100, 39)

In [59]:
seqs[3].shape

(100, 39)

In [60]:
ts = jnp.array(data['ts'])

In [61]:
ts.shape

(22567,)

In [62]:
ts

Array([1293, 1193, 1093, ...,  206,  106,    7], dtype=int32)

In [63]:
ts[4]

Array(893, dtype=int32)

In [64]:
h_ws = jnp.array(data['h_ws'])

In [68]:
h_ws[-1][:10, :10]

Array([[1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
       [1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
       [1., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
       [1., 1., 1., 1., 0., 0., 0., 0., 0., 0.],
       [1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [69]:
mask = jnp.array(data['mask'])

In [72]:
mask[0]

Array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]], dtype=float32)

In [74]:
h_tgt = jnp.array(data['h_tgt'])

In [78]:
h_tgt[-1][:10, :10]

Array([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [79]:
ts[-1]

Array(7, dtype=int32)